In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Reproducibility for consistent losses
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size, learning_rate, num_epochs, latent_dim = 128, 0.0002, 50, 100
image_size = 28 * 28

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_loader = DataLoader(datasets.MNIST('./data', True, transform, download=True), batch_size, shuffle=True)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, image_size), nn.Tanh())
    def forward(self, z): return self.model(z)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(image_size, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, img): return self.model(img.view(img.size(0), -1))

G = Generator().to(device)
D = Discriminator().to(device)
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=learning_rate, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=learning_rate, betas=(0.5, 0.999))

print("Starting GAN training...")
for epoch in range(num_epochs):
    for step, (real_imgs, _) in enumerate(train_loader):
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        # Train Discriminator
        z = torch.randn(bs, latent_dim).to(device)
        fake = G(z)
        real_labels = torch.ones(bs, 1).to(device)
        fake_labels = torch.zeros(bs, 1).to(device)
        d_loss = criterion(D(real_imgs), real_labels) + criterion(D(fake.detach()), fake_labels)
        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        # Train Generator
        g_loss = criterion(D(G(torch.randn(bs, latent_dim).to(device))), real_labels)
        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

        # Progress print exactly as in lab manual (at step 200)
        if (step + 1) % 200 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{step+1}/{len(train_loader)}], D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}")
print("Training completed!")

# === SHOW SAMPLE GENERATED IMAGES ===
print("\nGenerating and visualizing 10 sample MNIST digits from the trained GAN...")
G.eval()
with torch.no_grad():
    z = torch.randn(10, latent_dim).to(device)
    fake_images = G(z)

fig, axes = plt.subplots(1, 10, figsize=(20, 2))
for i in range(10):
    img = fake_images[i].view(28, 28).cpu().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f"Fake {i+1}")
plt.tight_layout()
plt.show()
